# Climate Risk Intelligence: Annual City Forecasts

This notebook audits the annual climate records, evaluates forecast methods with a rolling-origin backtest, and produces seven-year forecasts for four climate indicators across the included U.S. cities.

**Forecast period:** 2019–2025  
**Training data:** annual city-level observations through 2018  
**Deliverable:** `forecast_2019_2025.csv`, created in the notebook's working directory

The notebook is organized to run from top to bottom. Backtest metrics use only observations available at each forecast origin; the final forecast uses 2018 as its origin.

## 1. Setup and reproducibility

The analysis uses a fixed random seed and downloads the annual dataset from the Hugging Face repository below. Required packages include NumPy, pandas, Matplotlib, seaborn, scikit-learn, LightGBM, `datasets`, and a parquet engine such as `pyarrow`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import Ridge
import lightgbm as lg

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100
RNG = 42
np.random.seed(RNG)

TARGETS = {
    'Avg Temp': 'Average Annual Temperature (F)',
    'ToT Prcp': 'Total Annual Precipitation (in)',
    'Max Tmax': 'Hottest Day of the Year (F)',
    'cdd': 'Cooling Degree Days (F-days)',
}
TCOLS = list(TARGETS)
HORIZON = 7
FIRST_FORECAST_YEAR = 2019

## 2. Load the annual dataset

The source contains annual indices through 2018. A local parquet cache avoids downloading and parsing the dataset again on subsequent runs.

In [ ]:
from datasets import load_dataset

REPO = "CMU-OSPO/protostar-climate-train-to-2018"
CACHE = Path('/tmp/annual.parquet')
if CACHE.exists():
  annual = pd.read_parquet(CACHE)
else:
  annual = load_dataset(REPO, "annual_indices", split="train").to_pandas()
cities = load_dataset(REPO, "city_info", split="train").to_pandas()
print(annual.shape, annual['Name'].nunique(), 'cities;', annual['Year'].min(), '-', annual['Year'].max())


## 3. Data audit and cleaning

Years with more than 60 missing daily observations for a relevant measure are treated as unreliable. The following cells replace infinities, apply this rule, and summarize coverage in the 1989–2018 reference window.

In [ ]:
MISS_TOL = 60 # days; a year with >60 missing days (~16%) is not trusted

df = annual.copy()
df = df.replace([np.inf, -np.inf], np.nan)

bad_t = (df['Miss Tmax'] > MISS_TOL) | (df['Miss Tmin'] > MISS_TOL)
bad_p = df['Miss Prcp'] > MISS_TOL

audit = pd.DataFrame({
    'raw_nan': df[TCOLS].isna().sum(),
    'flagged_by_missing_days': pd.Series({
        'Avg Temp': int(bad_t.sum()), 'Max Tmax': int(bad_t.sum()),
        'cdd': int(bad_t.sum()), 'ToT Prcp': int(bad_p.sum())}),
    })
display(audit)

for c in ['Avg Temp', 'Max Tmax', 'cdd']:
  df.loc[bad_t, c] = np.nan
df.loc[bad_p, 'ToT Prcp'] = np.nan

df = df.sort_values(['Name', 'Year']).reset_index(drop=True)
print('NaN after cleaning:'); display(df[TCOLS].isna().sum().to_frame('nan_count'))
print('Rows with all four targets valid:', int(df[TCOLS].notna().all(axis=1).sum()), 'of', len(df))

In [ ]:
# The last 30 years (the training window the baselines use) must be well-populated.
recent = df[df['Year'].between(1989, 2018)]
cov = recent.groupby('Name')[TCOLS].apply(lambda g: g.notna().mean()).min(axis=1)
print('Worst per-city coverage of valid values, 1989-2018:', round(cov.min(), 3))
print('Cities with < 80% coverage:', (cov < 0.8).sum())
print('Valid target rows in 2018:', df[df.Year == 2018][TCOLS].notna().sum().to_dict())

## 4. Exploratory analysis

These summaries describe record coverage, long-term anomalies, city-level trends, year-to-year variability, and common signals across cities. They provide context for interpreting the forecast errors; they are not an additional validation set.

In [ ]:
# how long is each city's record and where are the cities?
span = df.dropna(subset=['Avg Temp']).groupby('Name')['Year'].agg(['min', 'max', 'count'])
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].hist(span['count'], bins=25, color='steelblue')
ax[0].set_title('Valid years of temperature record per city')
ax[0].set_xlabel('years')
last = df[df.Year == 2018].dropna(subset=['Avg Temp'])
sc = ax[1].scatter(last['Lon'], last['Lat'], c=last['Avg Temp'], cmap='RdYlBu_r', s=40, edgecolor='k', linewidth=0.3)
plt.colorbar(sc, ax=ax[1], label='2018 Avg Temp (F)')
ax[1].set_title('210 cities, colored by 2018 average temperature')
ax[1].set_xlabel('Lon'); ax[1].set_ylabel('Lat')
plt.tight_layout(); plt.show()

In [ ]:
# 2.2 National-average time series (mean of city anomalies, so station mix does not bias it)
def anomaly_by_city(frame, col, base=(1951, 1980)):
    b = frame[frame.Year.between(*base)].groupby('Name')[col].mean()
    return frame[col] - frame['Name'].map(b)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for ax, (c, lab) in zip(axes.ravel(), TARGETS.items()):
    tmp = df[['Name', 'Year', c]].copy()
    tmp['anom'] = anomaly_by_city(df, c)
    nat = tmp.groupby('Year')['anom'].agg(['mean', 'count'])
    nat = nat[nat['count'] >= 100]
    ax.plot(nat.index, nat['mean'], color='grey', lw=1, label='national mean anomaly')
    ax.plot(nat.index, nat['mean'].rolling(10, center=True, min_periods=5).mean(), color='crimson', lw=2.5, label='10-yr mean')
    ax.axhline(0, color='k', lw=0.6); ax.set_title(lab + ' - anomaly vs 1951-1980')
    ax.legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# 2.3 Is the warming real and heterogeneous? Per-city OLS slope over the last 30 years (1989-2018), per decade
def slope_per_decade(g, col):
    g = g.dropna(subset=[col])
    if len(g) < 20:
        return np.nan
    return np.polyfit(g['Year'], g[col], 1)[0] * 10

rec = df[df.Year.between(1989, 2018)]
slopes = pd.DataFrame({c: rec.groupby('Name').apply(lambda g, c=c: slope_per_decade(g, c)) for c in TCOLS})
display(slopes.describe().T[['mean', 'std', 'min', '50%', 'max']].round(3))
print('Share of cities with positive 1989-2018 trend:'); display((slopes > 0).mean().round(2).to_frame('share_positive'))

fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
for ax, (c, lab) in zip(axes, TARGETS.items()):
    ax.hist(slopes[c].dropna(), bins=30, color='darkorange', edgecolor='white')
    ax.axvline(0, color='k', lw=1); ax.set_title(c + ' trend / decade')
plt.tight_layout(); plt.show()

In [ ]:
# 2.4 Where is the trend strongest? Map of per-city temperature trend
m = slopes[['Avg Temp']].join(df.groupby('Name')[['Lat', 'Lon']].first())
fig, ax = plt.subplots(figsize=(9, 5.5))
sc = ax.scatter(m['Lon'], m['Lat'], c=m['Avg Temp'], cmap='coolwarm', s=55, edgecolor='k', linewidth=0.3,
                vmin=-m['Avg Temp'].abs().max(), vmax=m['Avg Temp'].abs().max())
plt.colorbar(sc, label='°F per decade, 1989-2018')
ax.set_title('Local warming rate of average temperature'); ax.set_xlabel('Lon'); ax.set_ylabel('Lat')
plt.show()

In [ ]:
# 2.5 Noise floor: year-to-year variability around each city's own smooth trend.
# A forecast cannot beat this irreducible interannual noise on average, so it frames "good" MAE.
def detrended_abs_dev(g, col, win=30):
    g = g[g.Year >= 1989].dropna(subset=[col])
    if len(g) < 20:
        return np.nan
    p = np.polyfit(g['Year'], g[col], 1)
    return np.mean(np.abs(g[col] - np.polyval(p, g['Year'])))

floor = pd.DataFrame({c: df.groupby('Name').apply(lambda g, c=c: detrended_abs_dev(g, c)) for c in TCOLS})
print('Mean absolute deviation around each city\'s own 1989-2018 linear trend (an in-sample noise floor):')
display(floor.mean().round(3).to_frame('noise_floor_MAE'))

In [ ]:
# 2.6 Persistence: lag-1 autocorrelation of city anomalies (after removing each city's mean).
# High value => last year is informative; low => it is mostly noise.
ac = {}
for c in TCOLS:
    w = df[df.Year >= 1950].pivot(index='Year', columns='Name', values=c)
    w = w - w.mean()
    ac[c] = {f'lag{k}': np.nanmean([w[col].autocorr(k) for col in w.columns]) for k in (1, 2, 3, 5)}
display(pd.DataFrame(ac).T.round(3))

In [ ]:
# 2.7 Cross-city structure: do all cities move together in a given year (shared signal)?
# Share of variance of year-to-year anomalies explained by the national common component.
rows = []
for c in TCOLS:
    w = df[df.Year >= 1950].pivot(index='Year', columns='Name', values=c)
    a = w - w.mean()
    nat = a.mean(axis=1)
    r2 = np.nanmean([np.corrcoef(a[col].dropna(), nat.loc[a[col].dropna().index])[0, 1] ** 2 for col in a.columns if a[col].notna().sum() > 30])
    rows.append((c, r2))
display(pd.DataFrame(rows, columns=['metric', 'mean R2 with national mean']).set_index('metric').round(3))

## 5. Rolling-origin backtest and required baselines

The evaluation replays the forecasting task at origin years 1995–2011. At each origin, forecasts cover the next seven years and are scored only where observations exist. All baseline inputs are restricted to data available by the origin year.

The required baselines are last valid year, the trailing 30-year average, and a linear trend fitted to the trailing 30 years.

In [ ]:
# Wide matrices: rows = years, columns = cities
YEARS = np.arange(df.Year.min(), 2019)
CITIES = sorted(df.Name.unique())
W = {c: df.pivot(index='Year', columns='Name', values=c).reindex(index=YEARS, columns=CITIES) for c in TCOLS}
META = df.groupby('Name')[['Lat', 'Lon']].first().reindex(CITIES)

def last_valid(M, T):
    """Most recent non-NaN value at or before year T, per city."""
    return M.loc[:T].ffill().iloc[-1]

def baseline_last_year(M, T):
    return last_valid(M, T)

def baseline_avg30(M, T):
    return M.loc[T - 29:T].mean()

def baseline_trend30(M, T, min_pts=15):
    sub = M.loc[T - 29:T]
    x = sub.index.values.astype(float)
    out = {}
    for city in sub.columns:
        y = sub[city].values
        ok = ~np.isnan(y)
        if ok.sum() >= min_pts:
            out[city] = np.polyfit(x[ok], y[ok], 1)
        else:
            out[city] = (0.0, np.nan)
    return pd.Series(out)  # (slope, intercept) per city

def trend30_forecast(M, T, h, coefs=None):
    coefs = baseline_trend30(M, T) if coefs is None else coefs
    return pd.Series({c: coefs[c][0] * (T + h) + coefs[c][1] for c in M.columns})

# sanity check on the real 2018 origin for one city
for c in TCOLS:
    print(c, '| Aberdeen 2018 last-yr=%.1f avg30=%.1f trend30(2019)=%.1f' % (
        baseline_last_year(W[c], 2018)['Aberdeen'], baseline_avg30(W[c], 2018)['Aberdeen'],
        trend30_forecast(W[c], 2018, 1)['Aberdeen']))

In [ ]:
ORIGINS = list(range(1995, 2012))  # T+7 <= 2018

def run_baselines(origins=ORIGINS):
    recs = []
    for c in TCOLS:
        M = W[c]
        for T in origins:
            ly, a30 = baseline_last_year(M, T), baseline_avg30(M, T)
            coefs = baseline_trend30(M, T)
            for h in range(1, HORIZON + 1):
                y = M.loc[T + h]
                tr = trend30_forecast(M, T, h, coefs)
                recs.append(pd.DataFrame({
                    'target': c, 'origin': T, 'h': h, 'city': CITIES, 'y': y.values,
                    'Last Year': ly.values, '30-Yr Avg': a30.values, 'Linear Trend': tr.values}))
    return pd.concat(recs, ignore_index=True)

bt = run_baselines()
print(bt.shape)
BASE = ['Last Year', '30-Yr Avg', 'Linear Trend']

def mae_table(frame, methods, by=None):
    f = frame.dropna(subset=['y'] + methods)  # score only rows where every method produced a forecast
    g = f.groupby(by) if by else f.assign(_all=1).groupby('_all')
    return g.apply(lambda d: pd.Series({m: np.mean(np.abs(d[m] - d['y'])) for m in methods}))

res_base = mae_table(bt, BASE, by='target').loc[TCOLS]
display(res_base.round(3))

In [ ]:
# MAE by forecast horizon: does the ranking of baselines change with lead time?
by_h = mae_table(bt, BASE, by=['target', 'h'])
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
for ax, c in zip(axes, TCOLS):
    by_h.loc[c].plot(ax=ax, marker='o')
    ax.set_title(c); ax.set_xlabel('horizon (years)'); ax.set_ylabel('MAE')
plt.tight_layout(); plt.show()

## 6. Global models

The global panel combines cities and forecast horizons. Features are computed from each origin's historical data; training examples are eligible only when their target year is no later than the current evaluation origin. The models are Ridge regression, LightGBM, and a half-trend benchmark, with an ensemble included in the comparison.

### Feature and training design

- Each training row represents a city, forecast origin, and horizon from 1 to 7 years.
- The target is the future value relative to that city's trailing 30-year average.
- Features include recent anomalies, trend information, variability, location, horizon, and national-level recent anomalies.
- For a backtest forecast from year `T`, model training is limited to examples whose target year is at or before `T`, preventing future-data leakage.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

FEATS = ['ly_a', 'a5_a', 'a10_a', 'trend_off', 'slope', 'sd30', 'lat', 'lon', 'h', 'a30', 'nat_ly', 'nat_a5']
ALL_YEARS = np.arange(YEARS.min(), 2019 + HORIZON)

def build_panel(c, first_origin=1925):
    M = W[c].reindex(ALL_YEARS)
    recs = []
    for T in range(first_origin, 2019):
        a30, a10, a5 = M.loc[T - 29:T].mean(), M.loc[T - 9:T].mean(), M.loc[T - 4:T].mean()
        ly = last_valid(M.loc[:2018], T)
        sd = M.loc[T - 29:T].std()
        coefs = baseline_trend30(M.loc[:2018], T)
        slope = pd.Series({k: v[0] for k, v in coefs.items()})
        nat_ly, nat_a5 = (ly - a30).mean(), (a5 - a30).mean()
        for h in range(1, HORIZON + 1):
            tr = trend30_forecast(M.loc[:2018], T, h, coefs)
            recs.append(pd.DataFrame({
                'city': CITIES, 'origin': T, 'h': h, 'y': M.loc[T + h].values,
                'a30': a30.values, 'ly_a': (ly - a30).values, 'a5_a': (a5 - a30).values, 'a10_a': (a10 - a30).values,
                'trend_off': (tr - a30).values, 'slope': slope.values, 'sd30': sd.values,
                'lat': META['Lat'].values, 'lon': META['Lon'].values, 'nat_ly': nat_ly, 'nat_a5': nat_a5,
                'Last Year': ly.values, 'Linear Trend': tr.values, '30-Yr Avg': a30.values}))
    p = pd.concat(recs, ignore_index=True)
    p['target_year'] = p['origin'] + p['h']
    return p

PANEL = {c: build_panel(c) for c in TCOLS}
print({c: p.shape for c, p in PANEL.items()})

In [ ]:
LGB_PARAMS = dict(objective='l1', n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=100,
                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, random_state=RNG, verbose=-1, n_jobs=8)

def fit_predict(c, T, return_models=False):
    p = PANEL[c]
    train = p[(p.target_year <= T) & p['y'].notna() & p['a30'].notna() & (p.origin >= 1925)]
    test = p[p.origin == T].copy()
    ytr = (train['y'] - train['a30']).values
    ridge = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=10.0)).fit(train[FEATS], ytr)
    gbm = lg.LGBMRegressor(**LGB_PARAMS).fit(train[FEATS], ytr)
    test['Ridge'] = test['a30'] + ridge.predict(test[FEATS])
    test['LightGBM'] = test['a30'] + gbm.predict(test[FEATS])
    test['Half-Trend'] = 0.5 * (test['30-Yr Avg'] + test['Linear Trend'])
    test['Ensemble'] = (test['LightGBM'] + test['Ridge'] + test['Linear Trend']) / 3
    test['target'] = c
    out = test[['target', 'origin', 'h', 'city', 'y', 'Last Year', '30-Yr Avg', 'Linear Trend', 'Half-Trend', 'Ridge', 'LightGBM', 'Ensemble']]
    return (out, gbm) if return_models else out

In [ ]:
from tqdm.auto import tqdm
bt_all = pd.concat([fit_predict(c, T) for c in TCOLS for T in tqdm(ORIGINS, desc=c)], ignore_index=True)
METHODS = ['Last Year', '30-Yr Avg', 'Linear Trend', 'Half-Trend', 'Ridge', 'LightGBM', 'Ensemble']
print(bt_all.shape)

## 7. Backtest results and uncertainty

The tables and plots below compare pooled MAE, performance by horizon and era, city-level gains, and feature importance. Lower MAE is better. The paired bootstrap resamples forecast-origin blocks to reflect dependence among cities and horizons within an origin.

In [ ]:
res = mae_table(bt_all, METHODS, by='target').loc[TCOLS]
display(res.round(3).style.highlight_min(axis=1, color='#b7e4c7'))

best_base = res[BASE].min(axis=1)
skill = (1 - res[METHODS].div(best_base, axis=0)) * 100
print('Skill vs best baseline (%, positive = better):')
display(skill.round(1).style.background_gradient(cmap='RdYlGn', axis=None, vmin=-10, vmax=10))

In [ ]:
# Paired uncertainty: bootstrap over (origin) blocks, because errors within an origin year are strongly correlated.
def block_bootstrap_diff(frame, m_new, m_ref, n=500, seed=RNG):
    f = frame.dropna(subset=['y', m_new, m_ref])
    per = f.groupby('origin').apply(lambda d: pd.Series({
        'n': len(d),
        'd': np.sum(np.abs(d[m_new] - d['y'])) - np.sum(np.abs(d[m_ref] - d['y']))
    }))
    rng = np.random.default_rng(seed)
    idx = np.arange(len(per)); out = []
    for _ in range(n):
        s = rng.choice(idx, len(idx))
        out.append(per['d'].values[s].sum() / per['n'].values[s].sum())
    return np.percentile(out, [5, 50, 95])

rows = []
for c in TCOLS:
    f = bt_all[bt_all.target == c]
    for m in ['Ensemble', 'LightGBM', 'Ridge']:
        ref = res.loc[c, BASE].idxmin()
        lo, mid, hi = block_bootstrap_diff(f, m, ref)
        rows.append((c, m, ref, lo, mid, hi, 'yes' if hi < 0 else ('no (worse)' if lo > 0 else 'inconclusive')))
boot = pd.DataFrame(rows, columns=['target', 'model', 'vs best baseline', 'dMAE 5%', 'dMAE med', 'dMAE 95%', 'significantly better?'])
display(boot.round(3))

In [ ]:
# Bar chart of pooled MAE
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
colors = ['#adb5bd'] * 3 + ['#74c0fc', '#4dabf7', '#1c7ed6', '#d9480f']
for ax, c in zip(axes, TCOLS):
    ax.bar(METHODS, res.loc[c, METHODS], color=colors)
    ax.set_title(c); ax.set_ylabel('MAE'); ax.tick_params(axis='x', rotation=70)
    ax.set_ylim(res.loc[c, METHODS].min() * 0.9, res.loc[c, METHODS].max() * 1.03)
plt.suptitle('Rolling-origin backtest MAE (origins 1995-2011, horizons 1-7)', y=1.03)
plt.tight_layout(); plt.show()

In [ ]:
# MAE by horizon for all methods
bh = mae_table(bt_all, METHODS, by=['target', 'h'])
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, c in zip(axes, TCOLS):
    for m in METHODS:
        ax.plot(bh.loc[c].index, bh.loc[c, m], marker='o', ms=3, lw=2.5 if m == 'Ensemble' else 1.2,
                ls='--' if m in BASE else '-', label=m)
    ax.set_title(c); ax.set_xlabel('horizon (yrs)')
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Does skill depend on the era? Split origins into early (1995-2003) and late (2004-2011)
bt_all['era'] = np.where(bt_all.origin <= 2003, '1995-2003 origins', '2004-2011 origins')
era = mae_table(bt_all, METHODS, by=['target', 'era'])
display(era.round(3))

In [ ]:
# Where do models help? Per-city MAE improvement of Ensemble vs best baseline, mapped.
rows = []
for c in TCOLS:
    f = bt_all[bt_all.target == c].dropna(subset=['y', 'Ensemble'] + BASE)
    ref = res.loc[c, BASE].idxmin()
    g = f.groupby('city').apply(lambda d: np.mean(np.abs(d[ref] - d['y'])) - np.mean(np.abs(d['Ensemble'] - d['y'])))
    rows.append(g.rename(c))
gain = pd.concat(rows, axis=1).join(META)
fig, axes = plt.subplots(1, 4, figsize=(19, 4))
for ax, c in zip(axes, TCOLS):
    v = gain[c].abs().quantile(0.95)
    sc = ax.scatter(gain['Lon'], gain['Lat'], c=gain[c], cmap='RdYlGn', vmin=-v, vmax=v, s=22)
    ax.set_title(f'{c}: MAE gain of Ensemble\n(green = better than best baseline)'); plt.colorbar(sc, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance of LightGBM trained with all data through 2011 (the last backtest origin)
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
for ax, c in zip(axes, TCOLS):
    _, g = fit_predict(c, 2011, return_models=True)
    imp = pd.Series(g.booster_.feature_importance('gain'), index=FEATS).sort_values()
    ax.barh(imp.index, imp.values / imp.sum(), color='teal'); ax.set_title(c)
plt.tight_layout(); plt.show()

## 8. Final forecast: 2019–2025

The final prediction uses 2018 as the forecast origin and the same feature and model pipeline as the backtest. The forecast table selects the lowest-MAE method by target based on the rolling-origin backtest. Cities without adequate recent observations receive the documented neighbor-based fallback below; this is an approximation, not a city-specific validated model.

In [ ]:
final = pd.concat([fit_predict(c, 2018) for c in TCOLS], ignore_index=True)
final['year'] = final['origin'] + final['h']

### Sparse-station fallback

* **Huntsville** has valid data only for 1895-1957 and 2018 (the station record is empty 1958-2017), so no baseline or model can use a 30-year climatology directly.
* `forecast = mean(forecast of 3 nearest well-observed cities) + offset`, where `offset` is the city's average difference from those neighbors over overlapping historical years.
* This is an explicit, documented approximation. Its accuracy cannot be backtested for Huntsville itself, but it is the same logic as spatial interpolation.

In [ ]:
def sparse_cities(min_valid=15):
    out = set()
    for c in TCOLS:
        m = W[c].loc[1989:2018].notna().sum()
        out |= set(m[m < min_valid].index)
    return sorted(out)

SPARSE = sparse_cities()
print('Sparse cities:', SPARSE)

def fill_sparse(final, k=3):
    final = final.copy()
    good = [c for c in CITIES if c not in SPARSE]
    for city in SPARSE:
        d = np.hypot(META.loc[good, 'Lat'] - META.loc[city, 'Lat'],
                     (META.loc[good, 'Lon'] - META.loc[city, 'Lon']) * np.cos(np.radians(META.loc[city, 'Lat'])))
        nb = d.nsmallest(k).index.tolist()
        print(f'{city}: neighbors = {nb}')
        for c in TCOLS:
            hist = W[c][city]
            offset = (hist - W[c][nb].mean(axis=1)).dropna().mean()
            for m in METHODS:
                mbf = final[(final.target == c) & final.city.isin(nb)].groupby('h')[m].mean()
                rows = (final.target == c) & (final.city == city)
                final.loc[rows, m] = final.loc[rows, 'h'].map(mbf) + offset
    return final

final = fill_sparse(final)
assert final[METHODS].notna().all().all()

best_method = res[METHODS].idxmin(axis=1)
print('Best backtest method per target:'); display(best_method.to_frame('method'))

sub = []
for c in TCOLS:
    f = final[final.target == c]
    sub.append(pd.DataFrame({'city': f['city'], 'year': f['year'], 'target': c, 'forecast': f[best_method[c]]}))
submission = pd.concat(sub).pivot_table(index=['city', 'year'], columns='target', values='forecast').reset_index()
submission = submission[['city', 'year'] + TCOLS]
assert len(submission) == len(CITIES) * HORIZON
assert submission['year'].between(FIRST_FORECAST_YEAR, FIRST_FORECAST_YEAR + HORIZON - 1).all()
assert submission['city'].nunique() == len(CITIES)
assert submission[TCOLS].notna().all().all()
print(submission.shape, '| NaNs:', int(submission[TCOLS].isna().sum().sum()))
display(submission.head(7))
submission.to_csv('forecast_2019_2025.csv', index=False)

### Forecast validation

The following checks confirm the requested city-year coverage, complete forecast values, and nonnegative precipitation and cooling-degree-day values. The saved CSV uses one row per city and forecast year.

In [ ]:
# Sanity: forecasts must be physically plausible and close to recent history
chk = submission.merge(df[df.Year == 2018][['Name'] + TCOLS].rename(columns={'Name': 'city'}), on='city', suffixes=('', '_2018'))
for c in TCOLS:
    d = (chk[c] - chk[c + '_2018']).abs()
    print(f'{c:9s} forecast range [{chk[c].min():8.1f}, {chk[c].max():8.1f}] mean |forecast - 2018| = {d.mean():6.2f}')
assert (submission['ToT Prcp'] >= 0).all() and (submission['cdd'] >= 0).all()

In [ ]:
# National-average trajectory: history + backtested model + 2019-2025 forecast
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (c, lab) in zip(axes.ravel(), TARGETS.items()):
    hist = df[df.Year >= 1960].groupby('Year')[c].mean()
    ax.plot(hist.index, hist.values, color='k', lw=1.3, label='observed (mean of cities)')
    for m, col in [('30-Yr Avg', '#868e96'), ('Linear Trend', '#f08c00'), (best_method[c], '#d9480f')]:
        fm = final[final.target == c].groupby('year')[m].mean()
        ax.plot(fm.index, fm.values, color=col, lw=2.5 if m == best_method[c] else 1.5, marker='o', ms=3,
                ls='--' if m in BASE else '-', label=f'forecast: {m}')
    ax.axvline(2018.5, color='gray', ls=':'); ax.set_title(lab); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Example cities: observed history and the 2019-2025 forecast with a simple empirical band
# (band = +/- the median absolute backtest error of the chosen method at that horizon)
EXAMPLES = [n for n in ['Phoenix', 'Miami', 'Seattle', 'Chicago'] if n in CITIES][:4]
fig, axes = plt.subplots(len(TCOLS), len(EXAMPLES), figsize=(4.3 * len(EXAMPLES), 3 * len(TCOLS)), sharex=True)
for i, c in enumerate(TCOLS):
    err = bt_all[bt_all.target == c].assign(e=lambda d: (d[best_method[c]] - d['y']).abs()).groupby('h')['e'].median()
    for j, city in enumerate(EXAMPLES):
        ax = axes[i, j]
        h = df[(df.Name == city) & (df.Year >= 1980)]
        ax.plot(h.Year, h[c], color='k', lw=1)
        f = final[(final.target == c) & (final.city == city)].sort_values('year')
        ax.plot(f.year, f[best_method[c]], color='#d9480f', lw=2, marker='o', ms=3)
        band = f['h'].map(err).values
        ax.fill_between(f.year, f[best_method[c]] - band, f[best_method[c]] + band, color='#d9480f', alpha=0.2)
        if i == 0: ax.set_title(city)
        if j == 0: ax.set_ylabel(c)
plt.tight_layout(); plt.show()

In [ ]:
# Backtest calibration: observed vs predicted scatter for the chosen method, horizon 1 and 7
fig, axes = plt.subplots(2, 4, figsize=(17, 8))
for j, c in enumerate(TCOLS):
    for i, hh in enumerate([1, 7]):
        f = bt_all[(bt_all.target == c) & (bt_all.h == hh)].dropna(subset=['y', best_method[c]])
        ax = axes[i, j]
        ax.scatter(f[best_method[c]], f['y'], s=3, alpha=0.25)
        lo, hi = f['y'].min(), f['y'].max(); ax.plot([lo, hi], [lo, hi], 'r--', lw=1)
        ax.set_title(f'{c}, h={hh}, {best_method[c]}'); ax.set_xlabel('predicted'); ax.set_ylabel('actual')
plt.tight_layout(); plt.show()

## 9. Interpretation

The backtest indicates modest improvements over the required baselines for average temperature and cooling degree days; the 30-year average remains the strongest method for total precipitation and hottest-day temperature. The historical trend analysis also shows positive 1989–2018 trends for most cities in average temperature and cooling degree days.

These are retrospective results over a limited set of forecast origins, not guarantees of future accuracy. The error tables and origin-block bootstrap should be considered alongside the point estimates. In particular, a forecast-error threshold is a screening heuristic rather than a formal probability of impact.

In [ ]:
cdd_best = best_method['cdd']
f_cdd = final[final.target == 'cdd'].groupby('city')[cdd_best].mean().rename('cdd_2019_25')
base_cdd = W['cdd'].loc[1989:2018].mean().rename('cdd_1989_2018')
pol = pd.concat([f_cdd, base_cdd, META], axis=1)
pol['increase'] = pol['cdd_2019_25'] - pol['cdd_1989_2018']
pol['increase_pct'] = 100 * pol['increase'] / pol['cdd_1989_2018']

# Typical error of the chosen method, averaged over a 7-yr window (errors partly average out over 7 years, so this is conservative)
mae_cdd = res.loc['cdd', cdd_best]
pol['above_error_threshold'] = pol['increase'] > mae_cdd
print(f'{cdd_best} CDD backtest MAE = {mae_cdd:.0f}; cities with forecast increase above that: {int(pol.above_error_threshold.sum())} / {len(pol)}')
display(pol.sort_values('increase', ascending=False).head(15)[['cdd_1989_2018', 'cdd_2019_25', 'increase', 'increase_pct', 'above_error_threshold']].round(1))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
sc = ax[0].scatter(pol['Lon'], pol['Lat'], c=pol['increase'], cmap='OrRd', s=20 + 80 * pol['above_error_threshold'], edgecolor='k', linewidth=0.3)
plt.colorbar(sc, ax=ax[0], label='forecast CDD increase vs 1989-2018')
ax[0].set_title('Expected extra cooling demand, 2019-2025 (large dots = above model error)')
ax[0].set_xlabel('Lon'); ax[0].set_ylabel('Lat')
top = pol.sort_values('increase', ascending=False).head(15).iloc[::-1]
ax[1].barh(top.index, top['increase'], color=np.where(top['above_error_threshold'], '#d9480f', '#adb5bd'))
ax[1].axvline(mae_cdd, color='k', ls='--', label='backtest MAE'); ax[1].legend()
ax[1].set_title('15 cities with the largest forecast CDD increase')
plt.tight_layout(); plt.show()

## 10. Additional data recommendations

The weather record alone says *how much* cooling will be needed, not *who is hurt by it* or *whether the forecast can be improved*. Recommended sources, in priority order:

| Data | Why it helps | Use |
|---|---|---|
| **ENSO / ocean-state indices** (NOAA ONI, PDO, AMO) | Largest known driver of year-to-year US temperature and precipitation swings; known months ahead | Forecast features |
| **Global mean temperature and CO2 / radiative forcing** (NASA GISTEMP, NOAA Mauna Loa) | Gives the long-term warming trend a physical anchor instead of a 30-year rolling window | Prior for city trend, physical anchor |
| **Dynamical climate model outputs** (CMIP6 downscaled, e.g. LOCA2 or NEX-GDDP) | Captures regional trend differences the local record can't | Prior for city trend, long-term bounds |
| **Gridded reanalyses** (ERA5, PRISM, nClimGrid) | Fills station gaps (e.g. Huntsville 1958-2017) and removes station-move artifacts | Cleaner targets, better sparse-city handling |
| **Urban form** (NLCD impervious surface, tree canopy, NASA/Landsat land-surface temperature) | Urban heat island changes local heat exposure beyond airport-station readings | Spatial downscaling to city neighborhoods |
| **Socioeconomic and housing** (ACS income, age 65+, A/C penetration from EIA RECS, housing age, CDC PLACES health data) | Defines who is vulnerable to a given CDD increase | Policy targeting and resource allocation |
| **Energy data** (EIA monthly electricity by state/utility, utility disconnection records) | Converts CDD into actual cost burden and grid stress | Policy sizing and trigger calibration |
| **Station metadata** (move history, instrument changes) | The Huntsville gap and the pre/post-1950 precision change in the early rows suggest inhomogeneities | Data QA/QC and homogenization |

Data quality items found in this notebook that deserve upstream fixes: years with essentially no readings stored as `-inf` / `0.0` instead of missing; Huntsville with a 60-year gap.

## 11. Policy proposal: forecast-triggered cooling assistance (the "CDD Trigger")

**Problem:** Cooling demand is rising in most cities (about 94% show a positive CDD trend over 1989-2018, with a typical increase of roughly 90 CDD per decade). Building air-conditioning and grid capacity is growing fastest. The hottest day of the year and rainfall are not predictable years ahead (section 7), so any policy that depends on forecasting them would be built on noise.

**Proposal:** Federal and state energy-assistance funds (e.g. LIHEAP's cooling component and state weatherization programs) allocate a share of money by a **forecast-triggered mechanism**:

1. Each year, compute each city's 7-year CDD forecast (this pipeline) and its increase over the 30-year baseline.
2. Cities whose forecast increase exceeds the pipeline's published backtest error are **triggered**; this keeps the policy from reacting to noise.
3. Triggered cities receive multi-year funding for: (a) efficient heat-pump/AC retrofits and weatherization for low-income and older-adult households, prioritized with utility bill-protection / no-shutoff rule during heat days.
4. Untriggered cities keep baseline funding. The index and error bar are published so the allocation is auditable.

**Why this design follows from the analysis:**
* It acts on CDD, the metric where the model measurably beats the baselines, and uses the trend-aware forecast rather than last year.
* The trigger threshold is the measured backtest error, so false alarms from noisy years are limited by construction.
* It avoids promising precision on rainfall or the single hottest day, where the best method is just the climatological average.
* Multi-year funding matches the forecast horizon; retrofits pay back over exactly the decade-scale trend we can detect.

**Evaluation plan:** Track per-household cooling cost burden and heat-related ER visits in triggered vs near-threshold untriggered cities (a regression-discontinuity design).

**Risks and Limits:** CDD is computed at one station per city (usually an airport), so urban heat island effects inside the city are missed; the trigger needs neighbor-smoothed adjustments. Funding formulas based on a model need transparency and appeals, especially for sparse-data stations like Huntsville, which here use a neighbor-based fallback.

## 12. Limitations and next steps

* Backtest origins end at 2011, so the most recent warming years are only partially reflected in the evaluation. The 2004-2011 era showed larger errors for every method.
* Only the annual indices were modeled. The daily table could add within-year information (e.g. seasonal anomalies) but is unlikely to help much for forecasts 1-7 years ahead.
* Max Tmax and Tot Prcp: the best method is the 30-year average; the next gain should come from ENSO / forcing features, not from more complex models on the same data.
* Model hyperparameters were fixed in advance, not tuned on the backtest, to avoid overfitting the evaluation. A proper tuning pass should use nested time-based validation.
* Station-level metrics, not area averages. A city's true experience differs from its airport station.